# Task 03 – Graph Data Preparation

Official splits, leakage-safe normalization, and bidirectional message-passing edges.

In [1]:
!pip install -q torch-geometric ogb streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 59.9 MB/s eta 0:00:00


In [2]:
import os
# Compatibility fix for trusted PyG objects downloaded by the official OGB package on PyTorch 2.6+.
os.environ['TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD'] = '1'

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/OGBN_Arxiv_Project')
RESULTS_ROOT = PROJECT_ROOT / 'results'
MODELS_DIR = PROJECT_ROOT / 'models'
SRC_DIR = PROJECT_ROOT / 'src'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'

for folder in [PROJECT_ROOT, RESULTS_ROOT, MODELS_DIR, SRC_DIR, ARTIFACTS_DIR,
               PROJECT_ROOT / 'notebooks', PROJECT_ROOT / 'visualizations',
               PROJECT_ROOT / 'dashboard', PROJECT_ROOT / 'report',
               PROJECT_ROOT / 'presentation', PROJECT_ROOT / 'video']:
    folder.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)

Mounted at /content/drive
Project root: /content/drive/MyDrive/OGBN_Arxiv_Project


In [4]:
# Common reproducibility settings
import warnings
import random
import numpy as np

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

In [5]:
# Libraries required for Task 03
import torch
from torch_geometric.utils import to_undirected

OUTPUT = RESULTS_ROOT / 'task03_data_preparation'
OUTPUT.mkdir(parents=True, exist_ok=True)

## 3.1 Official Training, Validation, and Test Splits

Uses the official OGB temporal splits and verifies that the three node groups do not overlap.

In [8]:
import pandas as pd
from ogb.nodeproppred import PygNodePropPredDataset

# Initialize the OGB dataset and get the number of nodes
dataset = PygNodePropPredDataset(name='ogbn-arxiv', root=OUTPUT)
N = dataset[0].num_nodes

# Official chronological splits supplied by OGB
split = dataset.get_idx_split()
train_idx = split['train'].view(-1)
valid_idx = split['valid'].view(-1)
test_idx = split['test'].view(-1)

split_table = pd.DataFrame({
    'split': ['train', 'validation', 'test'],
    'nodes': [len(train_idx), len(valid_idx), len(test_idx)]
})
split_table['percentage'] = 100 * split_table.nodes / N
display(split_table)

# Verify no overlap and complete coverage
assert len(set(train_idx.tolist()) & set(valid_idx.tolist())) == 0
assert len(set(train_idx.tolist()) & set(test_idx.tolist())) == 0
assert len(set(valid_idx.tolist()) & set(test_idx.tolist())) == 0
assert torch.cat([train_idx, valid_idx, test_idx]).unique().numel() == N
print('Official splits are complete and non-overlapping.')

Downloaded 0.08 GB: 100%|██████████| 81/81 [00:05<00:00, 16.06it/s]


Extracting /content/drive/MyDrive/OGBN_Arxiv_Project/results/task03_data_preparation/arxiv.zip


Processing...


Loading necessary files...
This might take a while.
Processing graphs...


100%|██████████| 1/1 [00:00<00:00, 12372.58it/s]


...

100%|██████████| 1/1 [00:00<00:00, 283.51it/s]

Saving...



Done!


,split,nodes,percentage
0,train,90941,53.702249
1,validation,29799,17.596830
2,test,48603,28.700921


Official splits are complete and non-overlapping.


## 3.2 Feature Preprocessing and Message-Passing Edges

The OGB node features are already numerical embeddings. Training-set mean and standard deviation are inspected, but the original features are retained because they produced the better validation result. Reverse edges are added for two-way GNN message passing; the original directed edges remain unchanged for graph analysis.

In [11]:
import json
from torch_geometric.utils import to_undirected

# Get the graph data object from the dataset
graph_data = dataset[0]

# Extract original features, labels, and directed edge index
x_original = graph_data.x
y = graph_data.y
edge_index_directed = graph_data.edge_index

# Inspect training-feature statistics without using validation or test data
train_mean = x_original[train_idx].mean(dim=0)
train_std = x_original[train_idx].std(dim=0, unbiased=False).clamp_min(1e-8)

# The original OGB embeddings are used after the preprocessing comparison
x = x_original.clone()

print('Missing feature values:', int(torch.isnan(x).sum()))
print('Mean of training features:', x[train_idx].mean().item())
print('Standard deviation of training features:', x[train_idx].std(unbiased=False).item())

# Add reverse edges only for GNN message passing
edge_index = to_undirected(edge_index_directed, num_nodes=N)
print('Original directed edges:', edge_index_directed.shape[1])
print('Message-passing edges:', edge_index.shape[1])

preprocessing_summary = {
    'split': 'Official OGB chronological split',
    'normalization': 'Original OGB embeddings retained after comparison',
    'label_shape': list(y.shape),
    'training_edges': 'Bidirectional edges created from original citations',
    'missing_values': int(torch.isnan(x).sum())
}

(OUTPUT / 'preprocessing_summary.json').write_text(json.dumps(preprocessing_summary, indent=2))
split_table.to_csv(OUTPUT / 'split_sizes.csv', index=False)

Missing feature values: 0
Mean of training features: 0.020849430933594704
Standard deviation of training features: 0.23353075981140137
Original directed edges: 1166243
Message-passing edges: 2315598


## 3.3 Preprocessing Decisions

- The official chronological OGB split provides a reproducible and leakage-safe evaluation.
- Labels are flattened to one dimension because cross-entropy loss expects one class index per node.
- The original 128-dimensional OGB embeddings are retained after comparing them with standardized features.
- Reverse edges support two-way message passing, while Task 02 keeps the original citation direction.